In [1]:
# Check GPU
!nvidia-smi

Tue Dec  9 13:27:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
HOME = os.getcwd()
print(f"[YOLO11-01-TRAINING] Home directory: {HOME}")

[YOLO11-01-TRAINING] Home directory: /kaggle/working


In [3]:
# Install ultralytics (YOLO11)
%pip install "ultralytics<=8.3.40" --quiet
%pip install "numpy<2" --quiet

# Prevent ultralytics from tracking activity
!yolo settings sync=False

import ultralytics
ultralytics.checks()

print("[YOLO11-01-TRAINING] Ultralytics installed successfully")

Ultralytics 8.3.40 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6567.2/8062.4 GB disk)
[YOLO11-01-TRAINING] Ultralytics installed successfully


In [4]:
import os
import shutil
import pandas as pd
import numpy as np
from pathlib import Path
import yaml
import json

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("[YOLO11-01-TRAINING] Libraries loaded")

[YOLO11-01-TRAINING] Libraries loaded


In [5]:
# Get HOME directory (defined in earlier cell)
if 'HOME' not in globals():
    HOME = os.getcwd()

# Dataset path (Kaggle)
DATASET_BASE_PATH = "/kaggle/input/dataset-sampah"

# Fallback for local testing
if not os.path.exists(DATASET_BASE_PATH):
    DATASET_BASE_PATH = input("Enter dataset path: ").strip()

# Class configuration
CLASSES = ['Organik', 'Anorganik', 'Lainnya']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}
IDX_TO_CLASS = {idx: cls for cls, idx in CLASS_TO_IDX.items()}

# YOLO11 configuration
YOLO_MODEL = 'yolo11s-cls.pt'  # YOLO11-Small Classification
EPOCHS = 100
IMGSZ = 224  # Same as MobileNetV3
BATCH_SIZE = 32

# Output directory
OUTPUT_DIR = './yolo11-01-training-output'
YOLO_DATASET_DIR = f'{HOME}/yolo11_dataset'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"[YOLO11-01-TRAINING] Configuration:")
print(f"  HOME directory: {HOME}")
print(f"  Dataset path: {DATASET_BASE_PATH}")
print(f"  Classes: {CLASSES}")
print(f"  Model: {YOLO_MODEL}")
print(f"  Epochs: {EPOCHS}")
print(f"  Image size: {IMGSZ}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  YOLO dataset directory: {YOLO_DATASET_DIR}")

[YOLO11-01-TRAINING] Configuration:
  HOME directory: /kaggle/working
  Dataset path: /kaggle/input/dataset-sampah
  Classes: ['Organik', 'Anorganik', 'Lainnya']
  Model: yolo11s-cls.pt
  Epochs: 100
  Image size: 224
  Batch size: 32
  Output directory: ./yolo11-01-training-output
  YOLO dataset directory: /kaggle/working/yolo11_dataset


In [6]:
from sklearn.model_selection import train_test_split

def scan_dataset(dataset_path):
    """Scan dataset and collect image paths"""
    data_info = []
    
    print("[YOLO11-01-TRAINING] Scanning dataset...")
    
    for class_name in CLASSES:
        class_path = os.path.join(dataset_path, class_name)
        
        if not os.path.exists(class_path):
            print(f"[WARNING] Folder not found: {class_path}")
            continue
        
        # Supported image extensions
        image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']
        image_files = []
        
        for ext in image_extensions:
            image_files.extend(Path(class_path).glob(f'*{ext}'))
            image_files.extend(Path(class_path).glob(f'*{ext.upper()}'))
        
        for img_path in image_files:
            data_info.append({
                'image_path': str(img_path),
                'class_name': class_name,
                'class_idx': CLASS_TO_IDX[class_name]
            })
    
    df = pd.DataFrame(data_info)
    
    print(f"\n[YOLO11-01-TRAINING] Scan complete:")
    print(f"  Total images: {len(df):,}")
    for class_name in CLASSES:
        count = (df['class_name'] == class_name).sum()
        print(f"  {class_name}: {count:,} ({count/len(df)*100:.1f}%)")
    
    return df

# Scan dataset
df_dataset = scan_dataset(DATASET_BASE_PATH)

[YOLO11-01-TRAINING] Scanning dataset...

[YOLO11-01-TRAINING] Scan complete:
  Total images: 18,000
  Organik: 6,000 (33.3%)
  Anorganik: 6,000 (33.3%)
  Lainnya: 6,000 (33.3%)


In [7]:
# Split dataset: 70% train, 15% val, 15% test
train_df, temp_df = train_test_split(
    df_dataset, 
    test_size=0.3, 
    stratify=df_dataset['class_idx'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['class_idx'],
    random_state=42
)

print(f"[YOLO11-01-TRAINING] Data split:")
print(f"  Train: {len(train_df):,} ({len(train_df)/len(df_dataset)*100:.1f}%)")
print(f"  Val:   {len(val_df):,} ({len(val_df)/len(df_dataset)*100:.1f}%)")
print(f"  Test:  {len(test_df):,} ({len(test_df)/len(df_dataset)*100:.1f}%)")

print(f"\nClass distribution - Train:")
for class_name in CLASSES:
    count = (train_df['class_name'] == class_name).sum()
    print(f"  {class_name}: {count:,} ({count/len(train_df)*100:.1f}%)")

print(f"\nClass distribution - Val:")
for class_name in CLASSES:
    count = (val_df['class_name'] == class_name).sum()
    print(f"  {class_name}: {count:,} ({count/len(val_df)*100:.1f}%)")

print(f"\nClass distribution - Test:")
for class_name in CLASSES:
    count = (test_df['class_name'] == class_name).sum()
    print(f"  {class_name}: {count:,} ({count/len(test_df)*100:.1f}%)")

[YOLO11-01-TRAINING] Data split:
  Train: 12,600 (70.0%)
  Val:   2,700 (15.0%)
  Test:  2,700 (15.0%)

Class distribution - Train:
  Organik: 4,200 (33.3%)
  Anorganik: 4,200 (33.3%)
  Lainnya: 4,200 (33.3%)

Class distribution - Val:
  Organik: 900 (33.3%)
  Anorganik: 900 (33.3%)
  Lainnya: 900 (33.3%)

Class distribution - Test:
  Organik: 900 (33.3%)
  Anorganik: 900 (33.3%)
  Lainnya: 900 (33.3%)


In [8]:
import shutil

def create_yolo_dataset_structure(train_df, val_df, test_df, output_dir):
    """
    Create YOLO classification dataset structure using symlinks
    """
    print(f"[YOLO11-01-TRAINING] Creating YOLO dataset structure at {output_dir}")
    
    # Remove existing directory if exists
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    
    # Create directory structure
    for split in ['train', 'val', 'test']:
        for class_name in CLASSES:
            os.makedirs(os.path.join(output_dir, split, class_name), exist_ok=True)
    
    # Create symlinks for train
    print("  Creating train symlinks...")
    for _, row in train_df.iterrows():
        src = row['image_path']
        dst = os.path.join(output_dir, 'train', row['class_name'], os.path.basename(src))
        
        # Use symlink on Linux/Mac, copy on Windows if symlink fails
        try:
            os.symlink(src, dst)
        except (OSError, NotImplementedError):
            shutil.copy2(src, dst)
    
    # Create symlinks for val
    print("  Creating val symlinks...")
    for _, row in val_df.iterrows():
        src = row['image_path']
        dst = os.path.join(output_dir, 'val', row['class_name'], os.path.basename(src))
        
        try:
            os.symlink(src, dst)
        except (OSError, NotImplementedError):
            shutil.copy2(src, dst)
    
    # Create symlinks for test
    print("  Creating test symlinks...")
    for _, row in test_df.iterrows():
        src = row['image_path']
        dst = os.path.join(output_dir, 'test', row['class_name'], os.path.basename(src))
        
        try:
            os.symlink(src, dst)
        except (OSError, NotImplementedError):
            shutil.copy2(src, dst)
    
    print(f"[YOLO11-01-TRAINING] YOLO dataset structure created")
    
    return output_dir

# Create dataset structure
yolo_dataset_path = create_yolo_dataset_structure(train_df, val_df, test_df, YOLO_DATASET_DIR)

[YOLO11-01-TRAINING] Creating YOLO dataset structure at /kaggle/working/yolo11_dataset
  Creating train symlinks...
  Creating val symlinks...
  Creating test symlinks...
[YOLO11-01-TRAINING] YOLO dataset structure created


In [9]:
# Verify dataset structure
print(f"[YOLO11-01-TRAINING] Dataset structure verification:")
for split in ['train', 'val', 'test']:
    print(f"\n{split.upper()}:")
    split_path = os.path.join(YOLO_DATASET_DIR, split)
    for class_name in CLASSES:
        class_path = os.path.join(split_path, class_name)
        count = len(list(Path(class_path).glob('*')))
        print(f"  {class_name}: {count} images")

[YOLO11-01-TRAINING] Dataset structure verification:

TRAIN:
  Organik: 4200 images
  Anorganik: 4200 images
  Lainnya: 4200 images

VAL:
  Organik: 900 images
  Anorganik: 900 images
  Lainnya: 900 images

TEST:
  Organik: 900 images
  Anorganik: 900 images
  Lainnya: 900 images


In [10]:
# Train YOLO11-Small classification model
print(f"\n[YOLO11-01-TRAINING] Starting training...")
print("="*60)

# IMPORTANT: For YOLO classification, 'data' parameter should point to the dataset root directory
# NOT the yaml file! The directory structure is: dataset/train/class1/, dataset/val/class2/, etc.

!yolo classify train \
    model={YOLO_MODEL} \
    data={YOLO_DATASET_DIR} \
    epochs={EPOCHS} \
    imgsz={IMGSZ} \
    batch={BATCH_SIZE} \
    patience=20 \
    save=True \
    plots=True \
    device=0 \
    workers=4 \
    pretrained=True \
    optimizer=Adam \
    lr0=0.001 \
    warmup_epochs=3

print("\n" + "="*60)
print(f"[YOLO11-01-TRAINING] Training completed!")


[YOLO11-01-TRAINING] Starting training...
100%|███████████████████████████████████████| 13.0M/13.0M [00:00<00:00, 138MB/s]
New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=classify, mode=train, model=yolo11s-cls.pt, data=/kaggle/working/yolo11_dataset, epochs=100, time=None, patience=20, batch=32, imgsz=224, save=True, save_period=-1, cache=False, device=0, workers=4, project=None, name=train, exist_ok=False, pretrained=True, optimizer=Adam, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buff

In [11]:
import glob
from IPython.display import Image as IPyImage, display

# Find latest training run
train_runs = glob.glob(f'{HOME}/runs/classify/train*')
if train_runs:
    latest_run = max(train_runs, key=os.path.getmtime)
    print(f"[YOLO11-01-TRAINING] Latest training run: {latest_run}")
    
    # Display results
    results_path = os.path.join(latest_run, 'results.png')
    if os.path.exists(results_path):
        print("\nTraining Results:")
        display(IPyImage(filename=results_path, width=800))
    
    # Display confusion matrix
    cm_path = os.path.join(latest_run, 'confusion_matrix_normalized.png')
    if os.path.exists(cm_path):
        print("\nNormalized Confusion Matrix:")
        display(IPyImage(filename=cm_path, width=600))
    
    # Display predictions
    val_batch_path = os.path.join(latest_run, 'val_batch0_pred.jpg')
    if os.path.exists(val_batch_path):
        print("\nValidation Batch Predictions:")
        display(IPyImage(filename=val_batch_path, width=800))
else:
    print("[WARNING] No training runs found!")

[YOLO11-01-TRAINING] Latest training run: /kaggle/working/runs/classify/train


In [12]:
from ultralytics import YOLO

# Find latest training run
train_runs = glob.glob(f'{HOME}/runs/classify/train*')
if train_runs:
    latest_run = max(train_runs, key=os.path.getmtime)
    best_model_path = os.path.join(latest_run, 'weights', 'best.pt')
    
    if os.path.exists(best_model_path):
        print(f"[YOLO11-01-TRAINING] Verifying trained model: {best_model_path}")
        
        # Load model
        model = YOLO(best_model_path)
        
        # Get model info
        print(f"\nModel info:")
        model.info()
        
        # Test prediction on sample image
        sample_img = train_df.iloc[0]['image_path']
        print(f"\nTesting prediction on: {sample_img}")
        print(f"True class: {train_df.iloc[0]['class_name']}")
        
        results = model.predict(sample_img, verbose=False)
        result = results[0]
        
        probs = result.probs.data.cpu().numpy()
        pred_idx = int(result.probs.top1)
        
        print(f"\nPrediction test:")
        print(f"  Probabilities shape: {probs.shape}")
        print(f"  Number of classes in model: {len(probs)}")
        print(f"  Expected number of classes: {len(CLASSES)}")
        print(f"  Predicted index: {pred_idx}")
        
        if len(probs) == len(CLASSES):
            print(f"  Predicted class: {CLASSES[pred_idx]}")
            print(f"  Confidence: {probs[pred_idx]:.4f}")
            print(f"\n✅ Model trained correctly with {len(CLASSES)} classes!")
        else:
            print(f"\n⚠️  WARNING: Model has {len(probs)} classes, but we expect {len(CLASSES)} classes!")
            print(f"  This indicates a training configuration issue.")
            print(f"  Please check the data.yaml file and retrain.")
    else:
        print(f"[ERROR] Model not found: {best_model_path}")
else:
    print("[ERROR] No training runs found!")

[YOLO11-01-TRAINING] Verifying trained model: /kaggle/working/runs/classify/train/weights/best.pt

Model info:
YOLO11s-cls summary: 151 layers, 5,446,851 parameters, 0 gradients, 12.1 GFLOPs

Testing prediction on: /kaggle/input/dataset-sampah/Anorganik/Anorganik3245.jpg
True class: Anorganik

Prediction test:
  Probabilities shape: (3,)
  Number of classes in model: 3
  Expected number of classes: 3
  Predicted index: 0
  Predicted class: Organik
  Confidence: 0.9910

✅ Model trained correctly with 3 classes!


In [13]:
# Copy best model to output directory
train_runs = glob.glob(f'{HOME}/runs/classify/train*')
if train_runs:
    latest_run = max(train_runs, key=os.path.getmtime)
    
    # Copy best weights
    best_model_src = os.path.join(latest_run, 'weights', 'best.pt')
    best_model_dst = os.path.join(OUTPUT_DIR, 'best_yolo11s_cls.pt')
    
    if os.path.exists(best_model_src):
        shutil.copy2(best_model_src, best_model_dst)
        model_size = os.path.getsize(best_model_dst) / (1024 * 1024)
        print(f"[YOLO11-01-TRAINING] Best model saved: {best_model_dst} ({model_size:.2f} MB)")
    
    # Copy last weights
    last_model_src = os.path.join(latest_run, 'weights', 'last.pt')
    last_model_dst = os.path.join(OUTPUT_DIR, 'last_yolo11s_cls.pt')
    
    if os.path.exists(last_model_src):
        shutil.copy2(last_model_src, last_model_dst)
        print(f"[YOLO11-01-TRAINING] Last model saved: {last_model_dst}")
    
    # Copy training plots
    for plot_name in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png']:
        plot_src = os.path.join(latest_run, plot_name)
        if os.path.exists(plot_src):
            shutil.copy2(plot_src, os.path.join(OUTPUT_DIR, plot_name))
            print(f"[YOLO11-01-TRAINING] Plot saved: {plot_name}")
    
    # Save training info
    training_info = {
        'model': YOLO_MODEL,
        'epochs': EPOCHS,
        'image_size': IMGSZ,
        'batch_size': BATCH_SIZE,
        'classes': CLASSES,
        'num_classes': len(CLASSES),
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'test_samples': len(test_df),
        'dataset_path': DATASET_BASE_PATH
    }
    
    with open(os.path.join(OUTPUT_DIR, 'training_info.json'), 'w') as f:
        json.dump(training_info, f, indent=2)
    
    print(f"[YOLO11-01-TRAINING] Training info saved")
else:
    print("[ERROR] No training runs found!")

[YOLO11-01-TRAINING] Best model saved: ./yolo11-01-training-output/best_yolo11s_cls.pt (31.40 MB)
[YOLO11-01-TRAINING] Last model saved: ./yolo11-01-training-output/last_yolo11s_cls.pt
[YOLO11-01-TRAINING] Training info saved


In [14]:
# Save split dataframes for evaluation
train_df.to_csv(f'{OUTPUT_DIR}/train_dataset.csv', index=False)
val_df.to_csv(f'{OUTPUT_DIR}/val_dataset.csv', index=False)
test_df.to_csv(f'{OUTPUT_DIR}/test_dataset.csv', index=False)

print(f"[YOLO11-01-TRAINING] Split datasets saved:")
print(f"  ✓ train_dataset.csv ({len(train_df):,} samples)")
print(f"  ✓ val_dataset.csv ({len(val_df):,} samples)")
print(f"  ✓ test_dataset.csv ({len(test_df):,} samples)")

[YOLO11-01-TRAINING] Split datasets saved:
  ✓ train_dataset.csv (12,600 samples)
  ✓ val_dataset.csv (2,700 samples)
  ✓ test_dataset.csv (2,700 samples)


In [15]:
summary = f"""
========================================
YOLO11 CLASSIFICATION - 01. TRAINING SUMMARY - JakOlah
========================================

Model: YOLO11-Small Classification
Task: Image Classification (3 classes)

Dataset:
  Total images: {len(df_dataset):,}
  Train: {len(train_df):,} ({len(train_df)/len(df_dataset)*100:.1f}%)
  Val:   {len(val_df):,} ({len(val_df)/len(df_dataset)*100:.1f}%)
  Test:  {len(test_df):,} ({len(test_df)/len(df_dataset)*100:.1f}%)

Classes: {', '.join(CLASSES)}

Training Configuration:
  Model: {YOLO_MODEL}
  Epochs: {EPOCHS}
  Image size: {IMGSZ}x{IMGSZ}
  Batch size: {BATCH_SIZE}
  Optimizer: Adam
  Learning rate: 0.001
  Patience: 20

Output Files:
  ✓ best_yolo11s_cls.pt
  ✓ last_yolo11s_cls.pt
  ✓ results.png
  ✓ confusion_matrix.png
  ✓ confusion_matrix_normalized.png
  ✓ training_info.json
  ✓ train_dataset.csv
  ✓ val_dataset.csv
  ✓ test_dataset.csv

Next Steps:
  1. Run yolo11-02-evaluation-step.ipynb to evaluate on test set
  2. Compare with MobileNetV3 and CNN-SVM models
  3. Analyze metrics and select best model

========================================
TRAINING COMPLETED ✅
========================================
"""

print(summary)

with open(f'{OUTPUT_DIR}/training_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary)

print(f"[YOLO11-01-TRAINING] Summary saved to {OUTPUT_DIR}/training_summary.md")


YOLO11 CLASSIFICATION - 01. TRAINING SUMMARY - JakOlah

Model: YOLO11-Small Classification
Task: Image Classification (3 classes)

Dataset:
  Total images: 18,000
  Train: 12,600 (70.0%)
  Val:   2,700 (15.0%)
  Test:  2,700 (15.0%)

Classes: Organik, Anorganik, Lainnya

Training Configuration:
  Model: yolo11s-cls.pt
  Epochs: 100
  Image size: 224x224
  Batch size: 32
  Optimizer: Adam
  Learning rate: 0.001
  Patience: 20

Output Files:
  ✓ best_yolo11s_cls.pt
  ✓ last_yolo11s_cls.pt
  ✓ results.png
  ✓ confusion_matrix.png
  ✓ confusion_matrix_normalized.png
  ✓ training_info.json
  ✓ train_dataset.csv
  ✓ val_dataset.csv
  ✓ test_dataset.csv

Next Steps:
  1. Run yolo11-02-evaluation-step.ipynb to evaluate on test set
  2. Compare with MobileNetV3 and CNN-SVM models
  3. Analyze metrics and select best model

TRAINING COMPLETED ✅

[YOLO11-01-TRAINING] Summary saved to ./yolo11-01-training-output/training_summary.md


In [16]:
# Create zip file of outputs
zip_filename = 'yolo11-01-training-output'
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)

print(f"[YOLO11-01-TRAINING] Output zipped to: {zip_filename}.zip")
print(f"  File size: {os.path.getsize(f'{zip_filename}.zip') / (1024*1024):.2f} MB")
print(f"\n💾 Download {zip_filename}.zip dari Kaggle output panel")
print(f"\nContents: Trained models, plots, training info, dataset splits")
print(f"\n[YOLO11-01-TRAINING] ✅ COMPLETED")

[YOLO11-01-TRAINING] Output zipped to: yolo11-01-training-output.zip
  File size: 46.49 MB

💾 Download yolo11-01-training-output.zip dari Kaggle output panel

Contents: Trained models, plots, training info, dataset splits

[YOLO11-01-TRAINING] ✅ COMPLETED
